# Lab 03 - Inheritance

This lab focuses on what is perhaps the most important aspect of object-oriented programming, that of *_inheritance_*, being the means by which we, as programmers:
- help reduce actual coding time by reducing the amount of code we must write
- reusing code wherever possible to also reduce errors when we would have previously rewritten code
- centralizing concepts in a hierarchy, making it easier to apply consistent changes

This will also support the concept in the next lab, that of _polymorphism_ - but we won't worry about that for now!

This lab will showcase how to form a basic hierarchy using the BankAccount concept we are already familiar with.  __We will give you a modified version of BankAccount as a starting point.__  This will _abstract_ the concepts common to three types of bank accounts:  a savings account (SAV), a chequing (CHQ) account, and an RRSP (retirement) account (RRS).  Each of these accounts have different attributes and procedures which we will provide to you in exercises below.  First, here's the abstract top-level BankAccount code, which we are giving you.  There have been some upgrades - there's now an account ID, with accessor and mutator, which can be a source for the comparators (now 'name', 'id', 'balance' as selectors to be used with _set_compare_by_); the constructor is similarly updated.  Also, the format of the *__str__* method has been changed (the reason for which will become clear later).

_**There is no execution result from this code; we would never actually make an instance of BankAccount again, because it is just the basis for "real" account types.



In [ ]:
# This is the top-level of an inheritance hierarchy.  This class will never be instantiated - it's too generic to be of direct use.
# However, it will allow us to store the common concepts of a BankAccount.  It also contains a minor enhancement, the addition of
# an account identification code, which can be alphanumeric.

class BankAccount(object): # this is the class header

    __compare_by = "name"

    def __init__(self, name, account_id, balance=0):
        self.__name = name
        self.__account_id = account_id
        self.__balance = balance
        self.__withdraw_override = False

    def set_compare_by(mode): # static method - NOW WITH ID AS A COMPARISON BASIS
        modes = ['name','id','balance']
        if mode in modes:
            BankAccount.__compare_by = mode
        else:
            print("Invalid comparator.")

    def get_balance(self):
        return self.__balance

    def set_id(self, account_id):
        self.__account_id = account_id

    def get_id(self):
        return self.__account_id

    def set_name(self, name):
        self.__name = name

    def get_name(self):
        return self.__name

    def enable_override(self):
        self.__withdraw_override = True

    def disable_override(self):
        self.__withdraw_override = False

    def deposit(self, amount):
        if isinstance(amount, int) or isinstance(amount, float):
            self.__balance += amount
        else:
            print("Balance unchanged.")

    def withdraw(self, amount):
        if isinstance(amount, int) or isinstance(amount, float):
            if not self.__withdraw_override and amount > self.__balance:
                print("Balance exceeded. Withdrawing available funds.")
                amount = self.__balance
            self.__balance -= amount
        else:
            print("Balance unchanged.")
        return amount

    def monthly_processing(self):
        pass # this is where account interest and fees will be added for savings and chequing accounts; fees only for retirement savings accounts

    def yearly_processing(self):
        pass # this is where account interest will be calculated on retirement savings accounts

    def __lt__(self, other):
        if isinstance(other, BankAccount):
            if BankAccount.__compare_by == "name":
                return self.__name < other.get_name()
            elif BankAccount.__compare_by == "id":
                return self.__account_id < other.get_id()
            else:
                return self.__balance < other.get_balance()

    def __eq__(self, other):
        if isinstance(other, BankAccount):
            if BankAccount.__compare_by == "name":
                return self.__name == other.get_name()
            elif BankAccount.__compare_by == "id":
                return self.__account_id == other.get_id()
            else:
                return self.__balance == other.get_balance()

    def __gt__(self, other):
        if isinstance(other, BankAccount):
            if BankAccount.__compare_by == "name":
                return self.__name > other.get_name()
            elif BankAccount.__compare_by == "id":
                return self.__account_id > other.get_id()
            else:
                return self.__balance > other.get_balance()

    def __le__(self, other):
        if isinstance(other, BankAccount):
            if BankAccount.__compare_by == "name":
                return self.__name <= other.get_name()
            elif BankAccount.__compare_by == "id":
                return self.__account_id <= other.get_id()
            else:
                return self.__balance <= other.get_balance()

    def __ge__(self, other):
        if isinstance(other, BankAccount):
            if BankAccount.__compare_by == "name":
                return self.__name >= other.get_name()
            elif BankAccount.__compare_by == "id":
                return self.__account_id >= other.get_id()
            else:
                return self.__balance >= other.get_balance()

    def __ne__(self, other):
        if isinstance(other, BankAccount):
            if BankAccount.__compare_by == "name":
                return self.__name != other.get_name()
            elif BankAccount.__compare_by == "id":
                return self.__account_id != other.get_id()
            else:
                return self.__balance != other.get_balance()

    def __str__(self):
        return_value = f"Name: {self.__name}\tID: {self.__account_id}\tBalance: {self.__balance}"
        return return_value


### READ THE ABOVE CODE CAREFULLY!

You need to understand the changes that were made to BankAccount.  It's not the same class you wrote in the previous lab.

You will note that there are two methods, *monthly_processing* and *yearly_processing* that have no code - just the keyword 'pass' which means 'do nothing'.  Why are these here?

These are _abstract methods_ that have no content, but are there to show programmers we expect stuff to happen in these methods.  They will be _overridden_ in the classes which will _extend_ BankAccount.  The comment hints at what the purpose is, but you cannot implement the action at the top hierarchical level, because "BankAccount" isn't a real account.  Thus, we save that behaviour for implementation later but we **note that it must be implemented in order for the account _subclasses_ derived from it to function correctly.**

\#\#\#\#\# _**RUN THIS MODULE NOW SO THAT IT IS AVAILABLE INSIDE THE PYTHON RUNTIME FOR THE SUBSEQUENT EXERCISES**_ \#\#\#\#\#

## Section 1 - Creating a Chequing Account

The first thing we need to generate is a class which _extends_ (that is, is based upon) the BankAccount class (shown above).  What behaviours will be different?

We need to add a _process_cheque_ method, which accepts an amount (int or float) as a parameter (verify the type), charges the user a per-cheque amount stored in a _local variable_ called _cheque_fee_ which should be set to 1.25 (but could be edited if the bank changes its chequing policy); it should then call _withdraw_ with the amount of the cheque.  Remember, _withdraw_ returns the actual amount withdrawn, which (unless overridden) is at most the amount of money in the account.  This method returns the amount successfully withdrawn.

What could go wrong?  Well, if you had \$50 in the chequing account, and you write a cheque for \$50, you get charged the cheque_fee _first_ - now you have \$48.75 in the account.  You can't honour the \$50 cheque!

Here's what happens:
- check the balance to be sure there's at least cheque_fee in the account - if not, do nothing (yes, I know, the bank would have a penalty for this, but we're keeping it simple)
- the cheque fee is charged
- the withdrawal is attempted - saving the withdrawn amount
- if that's less than what was requested, an _overdraft_ has occurred
- deposit the withdrawn amount _back into the account_ (return it) - but do not return the cheque processing fee.
- now, you will charge an NSF fee (non-sufficient funds penalty) to the account
-- enable override
-- charge the NSF fee which is allowed to overdraft the account because it's a bank penalty
-- disable override

You should make a local variable called _nsf_fee_ and set it to 40 - do this right after you define the cheque_fee

Note that you can call the deposit and withdraw methods from within the process_cheuqe method using self.deposit(amount) and self.withdraw(amount) calls.

Next, override the *monthly_processing* method to charge the monthly fee on chequeing accounts.  This method takes no parameters.  Again, use a local variable (probably called _monthly_fee_) to charge the $16.75 account maintenance fee.  Additionally, there's a \$25 overdraft penalty if the account is in overdraft (balance less than 0) at the time of monthly processing.  Set up a variable for this, too, called _overdraft_fee_.  Remember, this overdraft condition could be the result of insufficient money in the account to pay the monthly fees - so process it last.  All fee processing is done in override mode (don't forget to disable override afterwards!)  There is no value returned from this method.

You do not need to override the _yearly_processing_ method, as there are no actions that happen on a yearly basis with this account type.  The original method has `pass` as its only instruction, which has no effect whatsoever - this method also returns nothing.

**Note that you DO NOT HAVE TO REWRITE THE CLASS ABOVE.  EXTENSION MEANS YOU INHERIT THE CONTENTS OF THAT CLASS.  However, you do have to use the methods of the parent class because you cannot directly access the ivars of the parent class.  Parents don't allow children to raid their wallets, after all.**  We're supplying the header for this class for you, so you can see how it's set up.  The rest you must write.

**VERY IMPORTANT:** constructors (\_\_init__) are not inherited.  You must write an appropriate constructor, and then call the parent's constructor via `super().__init__(whatever,parameters,required)`

Finally, write a stringifier (\_\_str__) method that returns the account type, then a tab (\t), then the information from the BankAccount stringifier (called with `super().__str__()`) and then, if the account is currently in overdraft, adds a second line (with newline at the beginning) which says "ACCOUNT IN OVERDRAFT").  Remember that we don't add a newline at the end of the value, as that's up to the print statement or file output statement to do (that's using the stringifier). See the expected output for details.

In [ ]:
# We supply the header, you supply the rest.  Make sure you've carefully read all of the information from above.

class ChequingAccount(BankAccount): # Note that a ChequingAccount extends BankAccount - previously this was just 'object'

# now write your constructor and other methods as required


#### TEST CODE - DO NOT ALTER ####

ca = ChequingAccount("Chris","SB0031066838",100)
print(f"Attempting to process $85 cheque: {ca.process_cheque(85)}") # should return 85
print(f"Balance: {ca.get_balance()}")
print(f"\nAttempting to process $25 cheque: {ca.process_cheque(25)}") # should be NSF returning 0
print("\nAccount status:")
print(ca) # should show overdraft status
ca.monthly_processing() # do monthly processing
ca.yearly_processing() # do yearly processing
print("\nPost-processing status:")
print(ca)

## Section 1 - Expected Output

```
Attempting to process $85 cheque: 85
Balance: 13.75
Balance exceeded. Withdrawing available funds.

Attempting to process $25 cheque: 0

Account status:
CHEQUING ACCOUNT	Name: Chris	ID: SB0031066838	Balance: -27.5
ACCOUNT IS IN OVERDRAFT.

Post-processing status:
CHEQUING ACCOUNT	Name: Chris	ID: SB0031066838	Balance: -69.25
ACCOUNT IS IN OVERDRAFT.
```

Verify that the calculations you see are correct.  Also do them with a calculator or on paper - this is called a "hand check".  Do you see why these results are obtained?

It is _**our**_ responsibility, as programmers, to verify that the rules we write are being processed correctly.  Users will understand the processes, and how they did them manually, but they will not understand our coding, so we need to do all verifications carefully.

## Section 2 - Creating Savings Account

Next, you will create another class which extends BankAccount.  This time, no header is supplied, but you should be able to figure it out from what you just completed.

What behaviours will be different in _this_ class?

Savings accounts have the normal features of the original BankAccount, but they don't handle cheques.  So, all of the normal things are available here.

Savings accounts have an APR, or Annual Percentage Rate, that they pay yearly.  You will create an _instance variable_ to hold the APR, and name it appropriately, inside the _monthly_processing_ method.  As interest is paid monthly, the interest rate used in the monthly calculation will be 1/12 the APR.  Thus:  $interest_{monthly} = \frac{1}{12}interest_{yearly}$ - you should also set the monthly interest as a local variable.

Once a month, that interest is calculated on the current account balance, and that interest is then added to the account.  If left untouched, the interest next time will include the amount _plus_ the interest on the interest - that is, _compound interest_.  Savings accounts work best when money doesn't decrease in the account, but increases - so that interest can compound.  Of course, you can still withdraw money at any time because, hey, emergencies happen, right?

**The APR on this account type will be 5%.**  _Recall that percentages are stored as fractions normally._

There is, however, a catch.  Savings accounts with more than &dollar;1000 _before interest is added_ are not subject to a maintenance fee.  This is because the bank gives you an APR that is less than the APR that _they_ get, and that difference pays for the account (and more, banks are not in the business of losing money.)  If the balance before interest is added is _less than_ 1000, then the account is subject to a 2.50 maintenance fee.  The fee is much less than a chequing account because there are far fewer transactions expected on the account due to it being a savings account.  The maintenance fee is applied _after_ interest is calculated and added.  **That fee is applied in override mode so if the account had less than 2.50 after interest is added, it will be in overdraft.)  If at the end of these calculations the account is in overdraft, it gets a **_50 dollar overdraft fee_** - larger than the chequing account, because it pays less fees and is not designed for frequent transactions; you should set that as a local variable, and apply it if necessary. Again, the _monthly_processing_ method returns nothing.

Like the ChequingAccount, there are no fees levied annually so you don't have to override the _yearly_processing_ method - this method also returns nothing.

Remember, you are simply extending the original BankAccount class to create SavingsAccount - nothing more.  **_Only write what you have to._**  Remember, programmers are **lazy** (that is, we're efficient.)

Finally, write a stringifier (\_\_str__) method that returns the account type, then a tab (\t), then the information from the BankAccount stringifier (called with `super().__str__()`), then another tab, then the APR; and if in overdraft, the next line will say "ACCOUNT IS IN OVERDRAFT." - as shown in the expected output.

**IMPORTANT NOTE:** If the account is in overdraft before the interest calculation, interest is _not_ calculated and added (because it'd be negative interest).

In [ ]:
# You will write the entirety of your answer here - no starter code is provided.


#### TEST CODE - DO NOT ALTER ####

sa = SavingsAccount("Chris","SB0031065742",900)
print(f"Starting balance: ${sa.get_balance()}")
sa.deposit(200)
print(f"\nProcess $200 deposit, new balance ${sa.get_balance()}") # should return 1100
print("\nSix months go by with no additional deposits or withdrawals . . . ")
sa.monthly_processing()
print(f"Month 1 balance: ${sa.get_balance():.2f}")
sa.monthly_processing()
print(f"Month 2 balance: ${sa.get_balance():.2f}")
sa.monthly_processing()
print(f"Month 3 balance: ${sa.get_balance():.2f}")
sa.monthly_processing()
print(f"Month 4 balance: ${sa.get_balance():.2f}")
sa.monthly_processing()
print(f"Month 5 balance: ${sa.get_balance():.2f}")
sa.monthly_processing()
print(f"Month 6 balance: ${sa.get_balance():.2f}")
print("\nAccount status:")
print(sa)
print()
print("Withdrawing everything in the account!  PARTY TIME!")
sa.withdraw(sa.get_balance())
print("\nAccount status:")
print(sa)
print("\nProcessing monthly fees and interest . . . whoops.  That account's empty.")
sa.monthly_processing()
print(f"Month 7 balance: ${sa.get_balance():.2f}") # should show overdraft status
print("Ended month with a negative balance (maintenance fee, then overdraft fee).  Womp, womp.")
sa.yearly_processing() # do yearly processing
print("\nFinal post-processing status:")
print(sa)


## Section 2 - Expected Output

```
Starting balance: $900

Process $200 deposit, new balance $1100

Six months go by with no additional deposits or withdrawals . . .
Month 1 balance: $1104.58
Month 2 balance: $1109.19
Month 3 balance: $1113.81
Month 4 balance: $1118.45
Month 5 balance: $1123.11
Month 6 balance: $1127.79

Account status:
SAVINGS ACCOUNT	Name: Chris	ID: SB0031065742	Balance: 1127.7880547500479	APR: 5.0%

Withdrawing everything in the account!  PARTY TIME!

Account status:
SAVINGS ACCOUNT	Name: Chris	ID: SB0031065742	Balance: 0.0	APR: 5.0%

Processing monthly fees and interest . . . whoops.  That account's empty.
Month 7 balance: $-52.50
Ended month with a negative balance (maintenance fee, then overdraft fee).  Womp, womp.

Final post-processing status:
SAVINGS ACCOUNT	Name: Chris	ID: SB0031065742	Balance: -52.5	APR: 5.0%
ACCOUNT IS IN OVERDRAFT.
```

Yeah, taking everything out and failing to close the account was a dumb move which cost us a lot.

## Section 3 - Creating Registered Retirement Savings Account

Finally, you will create another class which extends BankAccount.  Again, no header is supplied, but you should be able to figure it out from what you just completed.

What behaviours will be different in _this_ class?

Retirement accounts have the normal features of the original BankAccount, and also don't handle cheques.  They have no monthly fees whatsoever.

However, they have an APR, and that's usually a lot bigger than a savings account - in this case, we'll assume 13%.

There are, however, special rules for these accounts:
- Any withdrawals must be tracked and reported because they constitute _taxable income_.  You have to create a variable at the appropriate level (instance or local) to accomplish this (figure it out!)  This means you'll need to override the _withdraw_ method, do everything it normally does, then add the actual amount withdrawn to the taxable income.  Remember, you can use super() to help you accomplish this!  Also remember that _withdraw_ returns the actual amount withdrawn.  You will need to add a parameter, 'taxable', because fees are not taxable but are deducted via withdraw.  If you set the default for the parameter to 'True' then you only need to supply it inside the _yearly_processing_ method to suppress taxation.
- Yearly processing consists of a similar process to the savings account monthly process:
1. If the balance is not negative, interest is calculated at the APR for this account - and this is added before any fees are taken.
2. This interest is not considered taxable income - but all withdrawals are, so it is not reported as taxable income.
3. An annual fee of $125 applies to this account, deducted after interest is applied.
4. If the account goes into overdraft at year end after the annual fee is applied, it is subject to a 150 dollar penalty, as this has to be reported to the Canada Revenue Agency.

So, there are no monthly fees, only yearly fees.

Finally, write a stringifier (\_\_str__) method that returns the account type, then a tab (\t), then the information from the BankAccount stringifier (called with `super().__str__()`), then another tab, then the APR; on the next line, it will print "TAXABLE INCOME: $nnn" where nnn is the amount of taxable income (withdrawals) made, presented as a float rounded to two decimal places.  If in overdraft, the next line will say "ACCOUNT IS IN OVERDRAFT." - as shown in the expected output.

In [ ]:
# again, you get no prototype code - just write your answer below


##### TESTING CODE - DO NOT ALTER #####
rs = RetirementSavings("Chris","SB0032824957", 150000)
print("Initial amount in account:")
print(rs)

print("\n10 years go by with an annual contribution of 10000 made before year-end...\n")
for i in range(1,11):
    rs.deposit(10000)
    rs.yearly_processing()
    print(f"Year {i}:")
    print(rs)
    print()

print("Time to withdraw it all and live my best life!")
party = rs.withdraw(rs.get_balance())
print("\nYear-end happens, but no deposit this year.")
rs.yearly_processing()
print("\nFinal account status:")
print(rs)
print(f"\nBut I don't care, because I'm off spending my ${party:.2f} (less income tax)\npartying someplace the weather is nice, and I won't be back!")

## Section 3 - Expected Output

```
Initial amount in account:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 150000	APR: 13.0%
TAXABLE INCOME: $0.00

10 years go by with an annual contribution of 10000 made before year-end...

Year 1:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 180675.0	APR: 13.0%
TAXABLE INCOME: $0.00

Year 2:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 215337.75	APR: 13.0%
TAXABLE INCOME: $0.00

Year 3:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 254506.6575	APR: 13.0%
TAXABLE INCOME: $0.00

Year 4:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 298767.52297499997	APR: 13.0%
TAXABLE INCOME: $0.00

Year 5:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 348782.30096175	APR: 13.0%
TAXABLE INCOME: $0.00

Year 6:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 405299.0000867775	APR: 13.0%
TAXABLE INCOME: $0.00

Year 7:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 469162.8700980586	APR: 13.0%
TAXABLE INCOME: $0.00

Year 8:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 541329.0432108062	APR: 13.0%
TAXABLE INCOME: $0.00

Year 9:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 622876.818828211	APR: 13.0%
TAXABLE INCOME: $0.00

Year 10:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: 715025.8052758785	APR: 13.0%
TAXABLE INCOME: $0.00

Time to withdraw it all and live my best life!

Year-end happens, but no deposit this year.

Final account status:
RETIREMENT SAVINGS	Name: Chris	ID: SB0032824957	Balance: -275.0	APR: 13.0%
TAXABLE INCOME: $715025.81
ACCOUNT IS IN OVERDRAFT.

But I don't care, because I'm off spending my $715025.81 (less income tax)
partying someplace the weather is nice, and I won't be back!
```

So, the account has an overdraft, and will continue to have it until they force it closed, but since I now live in a warm non-extradition country, I do _not_ care.

(Ok, I'd never do this, in real life. I'm a law abiding citizen.)

## END OF LAB

That's it for this lab. Next time we will look at _polymorphism_ where these classes will come back, along with the concept of a collection class that's aware of the data types in its collection.